1. Module install

In [0]:

# install world bank module

%pip install wbgapi

2. Fetch GDP data

In [0]:
import pandas as pd
import wbgapi as wb
from pyspark.sql import functions as F

# 1. Define country codes (USA, Euro Area 'EMU' for STOXX proxy, China, Japan, India)
countries = ["USA", "EMU", "CHN", "JPN", "IND"]

# 2. Indicator for GDP growth (annual percentage)
indicator = "NY.GDP.MKTP.KD.ZG"

# 3. Fetch data from 2000 to 2019 to match your stock data window
raw_gdp = wb.data.DataFrame(indicator, countries, time=range(2000, 2020))

# 4. Reset index so 'economy' (country code) becomes a standard column
gdp_df = raw_gdp.reset_index()

# 5. Convert the cleaned Pandas DataFrame to a Spark DataFrame
gdp_spark_df = spark.createDataFrame(gdp_df)

# 6. Add ingestion timestamp
gdp_spark_df = gdp_spark_df.withColumn("ingestion_timestamp", F.current_timestamp())

# 7. Write to your Bronze layer Delta table
(
    gdp_spark_df.write.format("delta").option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable("bronze.gdp_macro_raw")
)

print("GDP macro data successfully ingested into bronze.gdp_macro_raw!")